# MDE Viewpoint — Results Visualization

Loads the artifacts produced by the `experiments/` scripts and renders the final figures and tables for the report.

Sections:
1. Setup & data loading
2. Zero-shot comparison heatmap (seaborn)
3. Degradation-ratio bar charts per model
4. LoRA adaptation curves (N vs AbsRel) per category
5. Side-by-side qualitative panels: input | GT depth | model pred | error heatmap
6. Attention before vs after LoRA
7. Forgetting table: base vs LoRA on NYU + KITTI

In [ ]:
import json
import os
import sys
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT.parent))  # mde_viewpoint package import

with open(PROJECT_ROOT / 'configs' / 'config.yaml') as fh:
    CFG = yaml.safe_load(fh)
RESULTS_DIR = Path(CFG['results_dir']).resolve()
print('Results dir:', RESULTS_DIR)

## 1. Load all CSVs / JSONs from results/

In [ ]:
zero_shot_csv = RESULTS_DIR / 'zero_shot' / 'zero_shot_table.csv'
zero_shot_df = pd.read_csv(zero_shot_csv, index_col=0) if zero_shot_csv.exists() else pd.DataFrame()
print('Zero-shot table shape:', zero_shot_df.shape)
zero_shot_df.head()

## 2. Zero-shot comparison heatmap

In [ ]:
if not zero_shot_df.empty:
    abs_rel_cols = [c for c in zero_shot_df.columns if c.endswith('/abs_rel')]
    heat = zero_shot_df[abs_rel_cols].copy()
    heat.columns = [c.split('/')[0] for c in heat.columns]
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.heatmap(heat, annot=True, fmt='.3f', cmap='magma_r', ax=ax)
    ax.set_title('Zero-shot AbsRel by model x category (lower is better)')
    plt.tight_layout()
    plt.show()

## 3. Degradation-ratio bar charts

In [ ]:
if not zero_shot_df.empty:
    degr_cols = [c for c in zero_shot_df.columns if c.endswith('_degr') and 'abs_rel' in c]
    if degr_cols:
        degr = zero_shot_df[degr_cols].copy()
        degr.columns = [c.split('/')[0] for c in degr.columns]
        ax = degr.T.plot(kind='bar', figsize=(8, 4))
        ax.axhline(1.0, color='black', linestyle='--', alpha=0.6)
        ax.set_ylabel('Degradation ratio (cat AbsRel / Cat-A AbsRel)')
        ax.set_title('Per-category degradation relative to eye-level (Category A)')
        plt.tight_layout()
        plt.show()

## 4. LoRA adaptation curves

In [ ]:
lora_dir = RESULTS_DIR / 'lora'
fig, ax = plt.subplots(figsize=(8, 5))
if lora_dir.is_dir():
    for run_dir in sorted(lora_dir.iterdir()):
        curve = run_dir / 'adaptation_curve.csv'
        if curve.exists():
            df = pd.read_csv(curve)
            ax.plot(df['n_samples'], df['abs_rel'], marker='o', label=run_dir.name)
ax.set_xscale('log')
ax.set_xlabel('# fine-tune samples (N)')
ax.set_ylabel('AbsRel ↓')
ax.set_title('LoRA adaptation curves')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Qualitative panel: input | GT | pred | error

In [ ]:
from mde_viewpoint.analysis.error_heatmap import per_pixel_abs_rel, _denormalize_rgb
from mde_viewpoint.data.dataloader import load_category_dataloader
from mde_viewpoint.eval.metrics import apply_median_scaling
from mde_viewpoint.models.model_zoo import build_model
import torch

PANEL_MODEL = 'dav2'
PANEL_CAT = 'D'
PANEL_N = 4

loader = load_category_dataloader(
    split_dir=str(PROJECT_ROOT / 'data' / 'splits'),
    category=PANEL_CAT, subset='eval',
    image_size=int(CFG['image_size']),
    batch_size=PANEL_N, num_workers=0,
)
model = build_model(PANEL_MODEL, device=CFG['device'])
batch = next(iter(loader))
with torch.no_grad():
    pred = model.predict(batch['image']).cpu()
fig, axes = plt.subplots(PANEL_N, 4, figsize=(16, 4 * PANEL_N))
for i in range(PANEL_N):
    rgb = _denormalize_rgb(batch['image'][i])
    g = batch['depth'][i, 0].numpy()
    m = batch['mask'][i, 0].numpy().astype(bool)
    p = pred[i, 0].numpy()
    if not getattr(model, 'is_metric', False):
        p = apply_median_scaling(p, g, m)
    err = per_pixel_abs_rel(p, g, m)
    axes[i, 0].imshow(rgb); axes[i, 0].set_title('input'); axes[i, 0].axis('off')
    axes[i, 1].imshow(g, cmap='magma'); axes[i, 1].set_title('GT depth'); axes[i, 1].axis('off')
    axes[i, 2].imshow(p, cmap='magma'); axes[i, 2].set_title('pred'); axes[i, 2].axis('off')
    axes[i, 3].imshow(np.where(np.isfinite(err), err, 0), cmap='viridis', vmax=1.0)
    axes[i, 3].set_title('|pred-gt|/gt'); axes[i, 3].axis('off')
plt.tight_layout()
plt.show()

## 6. Attention maps before vs after LoRA

In [ ]:
from PIL import Image
before_files = sorted(glob(str(RESULTS_DIR / 'failure_analysis' / 'dav2' / PANEL_CAT / 'attention' / 'attn_*.png')))
after_files = sorted(glob(str(RESULTS_DIR / 'failure_analysis_lora' / 'dav2' / PANEL_CAT / 'attention' / 'attn_*.png')))
n_show = min(3, len(before_files), len(after_files))
if n_show > 0:
    fig, axes = plt.subplots(2, n_show, figsize=(5 * n_show, 8))
    if n_show == 1:
        axes = axes.reshape(2, 1)
    for i in range(n_show):
        axes[0, i].imshow(Image.open(before_files[i])); axes[0, i].axis('off')
        axes[0, i].set_title(f'before LoRA #{i}')
        axes[1, i].imshow(Image.open(after_files[i])); axes[1, i].axis('off')
        axes[1, i].set_title(f'after LoRA #{i}')
    plt.tight_layout()
    plt.show()
else:
    print('No before/after attention figures found yet.')

## 7. Forgetting table

In [ ]:
forget_path = RESULTS_DIR / 'forgetting' / 'forgetting.json'
if forget_path.exists():
    forget = json.loads(forget_path.read_text())
    rows = []
    for name in forget['results']['base']:
        for variant in ('base', 'lora'):
            r = forget['results'][variant].get(name, {})
            rows.append({'dataset': name, 'variant': variant,
                         'abs_rel': r.get('abs_rel'),
                         'rmse': r.get('rmse'),
                         'delta1': r.get('delta1')})
    df = pd.DataFrame(rows)
    display(df.pivot(index='dataset', columns='variant', values=['abs_rel', 'rmse', 'delta1']))
    deltas_df = pd.DataFrame(forget['deltas']).T
    print('\nDeltas (lora - base):')
    display(deltas_df)
else:
    print(f'No forgetting JSON found at {forget_path}')